In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import numpy as np
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from scipy.stats import norm

In [ ]:

plt.rcParams.update({
    'font.size': 22,  
    'font.weight': 'bold', 
    'axes.titlesize': 22, 
    'axes.labelsize': 22,
    'axes.titleweight': 'bold', 
    'axes.labelweight': 'bold',  
    'legend.fontsize': 22,

    # 添加全局线宽设置
    'axes.linewidth': 2,         # 坐标轴线宽
    'grid.linewidth': 1.5,      # 网格线宽
    'lines.linewidth': 2.5,     # 曲线线宽
    'xtick.major.width': 2,     # X轴主刻度线宽
    'ytick.major.width': 2,     # Y轴主刻度线宽
    'hatch.linewidth': 2,       # 填充图案线宽
})
# 设置设备
device = "cuda" if torch.cuda.is_available() else "cpu"


pi = 100.0  # 参数 pi
beta = 0.161  # 参数 beta
alpha = 0.332  # 参数 alpha
r = 0.05    # 无风险利率
T = 4.0     # 资产价格相关参数
tau = 3.0   # 期权到期时间
K = 50.0    # 行权价
# 定义模型结构
class simple_NN2(nn.Module):
    def __init__(self):
        super(simple_NN2, self).__init__()
        self.linear_tanh_stack = nn.Sequential(
            nn.Linear(2, 16),  # 2 inputs: A, t
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, A, t):
        x_stack = torch.cat([A, t], dim=1)  # concatenate A and t
        out = self.linear_tanh_stack(x_stack)
        return out

# 加载模型
model = torch.load("pinn_model_t   ure_carbon.pth")
model.to(device)  # 确保模型加载到正确设备
model.eval()      # 切换到评估模式

# 加载结果矩阵
data = np.load("sol_pinn_matrix.npz")
sol_pinn_matrix = torch.tensor(data["sol_pinn_matrix"]).to(device)

#绘图
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Create a surface plot
ax.plot_trisurf(sol_pinn_matrix[:,0].to("cpu").detach().numpy(), sol_pinn_matrix[:,1].to("cpu").detach().numpy(),
                sol_pinn_matrix[:,2].to("cpu").detach().numpy(), cmap='viridis')

# Add labels and title
ax.set_xlabel('A')
ax.set_ylabel('t')
ax.set_zlabel('C')
ax.set_title('PINN solution')

# 提取资产价格 A 和时间 t
A = sol_pinn_matrix[:, 0].unsqueeze(-1).to(device).requires_grad_()  # 资产价格 A
t = sol_pinn_matrix[:, 1].unsqueeze(-1).to(device).requires_grad_()  # 时间 t
C = sol_pinn_matrix[:, 2].unsqueeze(-1).to(device).requires_grad_()  # 期权价格 C





# 自动微分工具计算偏导数
# 自动求导工具
def df(output: torch.Tensor, input_var: torch.Tensor, order: int = 1) -> torch.Tensor:
    df_value = output
    for _ in range(order):
        df_value = torch.autograd.grad(
            df_value,
            input_var,
            grad_outputs=torch.ones_like(df_value),
            create_graph=True,
            retain_graph=True,
        )[0]
    return df_value


def dfdt(model: simple_NN2, A: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = model(A, t)
    return df(f_value, t, order=order)


def dfdx(model: simple_NN2, A: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = model(A, t)
    return df(f_value, A, order=order)

# 自动微分计算偏导数
dfdt_vals = dfdt(model, A, t)  # 时间导数
dfdx2_vals = dfdx(model, A, t, order=2)  # 二阶空间导数


In [ ]:
# 提取预测结果
A_np = sol_pinn_matrix[:, 0].to("cpu").detach().numpy()  # Asset price
t_np = sol_pinn_matrix[:, 1].to("cpu").detach().numpy()  # Time
C_np = sol_pinn_matrix[:, 2].to("cpu").detach().numpy()  # Option price

# 确定绘图的时间点 t（以 0.5 为间隔）
time_points = np.arange(0, tau + 0.1, 0.5)

# 创建绘图
plt.figure(figsize=(10, 10))
for t_fixed in time_points:
    # 提取当前时间点的价格和资产价格
    indices = np.isclose(t_np, t_fixed, atol=1e-2)  # 找到接近 t_fixed 的点
    A_at_t = A_np[indices]
    C_at_t = C_np[indices]

    # 绘制曲线
    plt.plot(A_at_t, C_at_t, label=f"t = {t_fixed:.1f}")

# 添加图例和标签
plt.xlabel("Asset Price (A)")
plt.ylabel("Option Price (C)")
# plt.title("Option Price vs Asset Price at Different Time Points")
plt.title("alpha=0.332 beta=0.161")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 创建网格
A_range = np.linspace(A_np.min(), A_np.max(), 250)
t_range = np.linspace(t_np.min(), t_np.max(), 250)
A_grid, t_grid = np.meshgrid(A_range, t_range)

# 插值计算网格上的期权价格
C_grid = griddata((A_np, t_np), C_np, (A_grid, t_grid), method='cubic')

# 绘制二维势能面
plt.figure(figsize=(10, 10))
contour = plt.contourf(A_grid, C_grid, t_grid, levels=50, cmap="viridis")

# 添加颜色条
cbar = plt.colorbar(contour)
cbar.set_label("Time (t)")

# 添加标签和标题
plt.xlabel("Asset Price (A)")
plt.ylabel("Option Price (C)")
# plt.title("2D Potential Surface of Option Price")
plt.title("alpha=0.332 beta=0.161")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# 确定绘图的时间点 t（以 0.5 为间隔）
time_points = np.arange(0, tau + 0.1, 0.5)

In [ ]:
# 创建一个图窗
fig = plt.figure(figsize=(21, 6))

# 第一个子图：3D表面图
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_trisurf(sol_pinn_matrix[:,0].to("cpu").detach().numpy(), sol_pinn_matrix[:,1].to("cpu").detach().numpy(),
                 sol_pinn_matrix[:,2].to("cpu").detach().numpy(), cmap='viridis')
ax1.set_xlabel('A')
ax1.set_ylabel('t')
ax1.set_zlabel('C')
ax1.set_title('PINN solution')

# 第二个子图：价格曲线图
ax2 = fig.add_subplot(132)
for t_fixed in time_points:
    indices = np.isclose(t_np, t_fixed, atol=1e-2)
    A_at_t = A_np[indices]
    C_at_t = C_np[indices]
    ax2.plot(A_at_t, C_at_t, label=f"t = {t_fixed:.1f}")
ax2.set_xlabel("Asset Price (A)")
ax2.set_ylabel("Option Price (C)")
ax2.set_title("Option Price vs Asset Price at Different Time Points")
ax2.legend()
ax2.grid(True)

# 第三个子图：2D势能面图
ax3 = fig.add_subplot(133)
contour = ax3.contourf(A_grid, C_grid, t_grid, levels=50, cmap="viridis")
cbar = plt.colorbar(contour, ax=ax3)
cbar.set_label("Time (t)")
ax3.set_xlabel("Asset Price (A)")
ax3.set_ylabel("Option Price (C)")
ax3.set_title("2D Potential Surface of Option Price")
ax3.grid(True, linestyle="--", alpha=0.5)

# 调整子图间的间距
plt.tight_layout()

# 显示图窗
plt.show()

In [ ]:

# 残差计算
Phi = torch.distributions.Normal(0, 1).cdf  # 标准正态分布 CDF
Phi_inv = torch.distributions.Normal(0, 1).icdf  # 标准正态分布反函数

z_t = beta * (T - t).pow(-alpha)
factor = (pi * Phi(Phi_inv(A / pi)) * torch.sqrt(z_t)).pow(2)
residual = dfdt_vals + 0.5 * factor * dfdx2_vals - r * model(A, t)

# 计算残差的均方误差
residual_mse = torch.mean(residual.pow(2))
print(f"Residual MSE: {residual_mse.item()}")


In [ ]:

# 可视化残差（使用两个子图：A vs Residual 和 t vs Residual）
residual = residual.detach().cpu().numpy()
A_vals = A.detach().cpu().numpy()
t_vals = t.detach().cpu().numpy()

# 创建一个图形，包含两个子图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# A 与残差的关系
ax1.scatter(A_vals, residual, label="Residual vs A", alpha=0.5, color="blue")
ax1.axhline(0, color="black", linestyle="--")
ax1.set_title("Residual vs Asset Price (A)")
ax1.set_xlabel("Asset Price (A)")
ax1.set_ylabel("Residual")
ax1.grid(True)
ax1.legend()

# t 与残差的关系
ax2.scatter(t_vals, residual, label="Residual vs t", alpha=0.5, color="red")
ax2.axhline(0, color="black", linestyle="--")
ax2.set_title("Residual vs Time (t)")
ax2.set_xlabel("Time (t)")
ax2.set_ylabel("Residual")
ax2.grid(True)
ax2.legend()

# 显示图形
plt.tight_layout()
plt.show()



In [ ]:



### **1. 检查边界条件**

# 边界条件 A = 0 和 A = 100
A_min = torch.zeros_like(t, device=device)  # 创建与 t 形状匹配的张量，值为 0
A_max = torch.ones_like(t, device=device) * 100  # 创建与 t 形状匹配的张量，值为 100

# 对于 A = 0 时的预测
C_at_A_min = model(A_min, t)
print(f"Boundary condition at A = 0: {C_at_A_min.mean().item()}")

# 对于 A = 100 时的预测
C_at_A_max = model(A_max, t)

# 计算预期值
expected_at_A_max = torch.maximum(A_max - K * torch.exp(-r * (tau - t)), torch.tensor(0.0, device=device))
print(f"Boundary condition at A = 100: {C_at_A_max.mean().item()} (Expected: {expected_at_A_max.mean().item()})")



### **2. 检查时间边界条件**

# 在 t = tau 时的模型预测
t_end = torch.full_like(A, tau, device=device)  # 将 t_end 扩展为与 A 形状一致

# 计算期权价格的最终条件 C_tau = max(A_tau - K, 0)
final_condition = torch.maximum(A - K, torch.tensor(0.0, device=device))

# 模型预测
C_at_t_end = model(A, t_end)
print(f"Boundary condition at t = tau: {C_at_t_end[:5].squeeze().detach().cpu().numpy()} (Expected: {final_condition[:5].squeeze().detach().cpu().numpy()})")

### **3. 检查初始条件**
# 在 t = 0 时的模型预测
t_start = torch.full_like(A, 0.0, device=device)  # 将 t_start 扩展为与 A 形状一致

# 模型预测
C_at_t_start = model(A, t_start)
print(f"Initial condition at t = 0: {C_at_t_start[:5].squeeze().detach().cpu().numpy()}")

In [ ]:
import pandas as pd

# 数据组织
data = {
    "Condition": [
        "Boundary (A=0)", 
        "Boundary (A=100)", 
        "Boundary (t=tau)", 
        "Initial (t=0)"
    ],
    "Value": [
        "-0.0118", 
        "54.9108 (Expected: 53.5690)", 
        "[-0.0128, -0.0128, -0.0128, -0.0128, -0.0128] (Expected: [0, 0, 0, 0, 0])", 
        "[-0.0249, -0.0249, -0.0249, -0.0249, -0.0249]"
    ]
}

# 创建DataFrame
df = pd.DataFrame(data)

# 打印表格
print(df)


In [ ]:
# 计算希腊字母
Delta = dfdx(model, A, t, order=1)  # Delta = ∂C/∂A
Gamma = dfdx(model, A, t, order=2)  # Gamma = ∂²C/∂A²
Theta = dfdt(model, A, t, order=1)  # Theta = ∂C/∂t

# 将结果转换为 NumPy（可选）
Delta_np = Delta.detach().cpu().numpy()
Gamma_np = Gamma.detach().cpu().numpy()
Theta_np = Theta.detach().cpu().numpy()

# 打印结果
print("Delta shape:", Delta_np.shape)
print("Gamma shape:", Gamma_np.shape)
print("Theta shape:", Theta_np.shape)

In [ ]:
# 确保资产价格 A 和期权价格 C 不为零（避免数值问题）
epsilon = 1e-8  # 防止除零


# 计算 Omega
Omega = (Delta * A) / (C + epsilon)  # 加 epsilon 防止 C = 0

# 将结果转换为 NumPy（可选）
Omega_np = Omega.detach().cpu().numpy()

# 打印结果
print("Omega shape:", Omega_np.shape)


In [ ]:
# 3D 绘图 Omega
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Create a surface plot for Omega
ax.plot_trisurf(A.detach().cpu().numpy().flatten(),
                t.detach().cpu().numpy().flatten(),
                Omega.detach().cpu().numpy().flatten(),
                cmap='viridis')

# Add labels and title
ax.set_xlabel('A')
ax.set_ylabel('t')
ax.set_zlabel('Omega')
ax.set_title('Option Elasticity (Omega)')
plt.show()


In [ ]:
# 3D 绘图 Delta
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Create a surface plot for Delta
ax.plot_trisurf(A.detach().cpu().numpy().flatten(),
                t.detach().cpu().numpy().flatten(),
                Delta.detach().cpu().numpy().flatten(),
                cmap='viridis')

# Add labels and title
ax.set_xlabel('A')
ax.set_ylabel('t')
ax.set_zlabel('Delta')
ax.set_title('Option Elasticity (Delta)')
plt.show()

In [ ]:
# 创建绘图
plt.figure(figsize=(10, 6))

for t_fixed in time_points:
    # 提取接近当前时间点的点
    indices = np.isclose(t_np, t_fixed, atol=1e-2)  # 找到接近 t_fixed 的点
    A_at_t = A_np[indices]
    Delta_at_t = Delta_np[indices]

    # 绘制曲线
    plt.plot(A_at_t, Delta_at_t, label=f"t = {t_fixed:.1f}")

# 添加图例和标签
plt.xlabel("Asset Price (A)")
plt.ylabel("Delta")
plt.title("Delta vs Asset Price at Different Time Points")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 创建绘图
plt.figure(figsize=(10, 6))

for t_fixed in time_points:
    # 提取接近当前时间点的点
    indices = np.isclose(t_np, t_fixed, atol=1e-2)  # 找到接近 t_fixed 的点
    A_at_t = A_np[indices]
    Gamma_at_t = Gamma_np[indices]
    plt.plot(A_at_t, Gamma_at_t, label=f"t = {t_fixed:.1f}")


# 添加图例和标签
plt.xlabel("Asset Price (A)")
plt.ylabel("Gamma")
plt.title("Gamma vs Asset Price at Different Time Points")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 创建绘图
plt.figure(figsize=(10, 6))

for t_fixed in time_points:
    # 提取接近当前时间点的点
    indices = np.isclose(t_np, t_fixed, atol=1e-2)  # 找到接近 t_fixed 的点
    A_at_t = A_np[indices]
    Theta_at_t = Theta_np[indices]
    plt.plot(A_at_t, Theta_at_t, label=f"t = {t_fixed:.1f}")



# 添加图例和标签
plt.xlabel("Asset Price (A)")
plt.ylabel("Theta")
plt.title("Theta vs Asset Price at Different Time Points")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 创建绘图
plt.figure(figsize=(10, 6))

for t_fixed in time_points:
    # 提取接近当前时间点的点
    indices = np.isclose(t_np, t_fixed, atol=1e-2)  # 找到接近 t_fixed 的点
    A_at_t = A_np[indices]
    Omega_at_t = Omega_np[indices]
    plt.plot(A_at_t, Omega_at_t, label=f"t = {t_fixed:.1f}")




# 添加图例和标签
plt.xlabel("Asset Price (A)")
plt.ylabel("Omega")
plt.title("Omega vs Asset Price at Different Time Points")
plt.legend()
plt.grid(True)
# 设置y轴的范围为-50到50
plt.ylim(-20, 20)
plt.show()

In [ ]:
Omega_at_t

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 绘制三维图
def plot_3d_surface(A_grid, t_grid, greek_vals, greek_name):
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    ax.plot_surface(A_grid, t_grid, greek_vals, cmap='viridis')

    ax.set_xlabel('Asset Price (A)')
    ax.set_ylabel('Time (t)')
    ax.set_zlabel(greek_name)
    ax.set_title(f'{greek_name} vs Asset Price (A) and Time (t)')
    
    plt.show()

# 绘制 Delta
plot_3d_surface(A_grid, t_grid, Delta_vals, 'Delta')

# 绘制 Gamma
plot_3d_surface(A_grid, t_grid, Gamma_vals, 'Gamma')

# 绘制 Theta
plot_3d_surface(A_grid, t_grid, Theta_vals, 'Theta')

# 绘制 Omega
plot_3d_surface(A_grid, t_grid, Omega_vals, 'Omega')


In [ ]:
# 插值计算网格上的希腊字母
Delta_grid = griddata((A_grid.flatten(), t_grid.flatten()), Delta_vals.flatten(), (A_grid, t_grid), method='cubic')
Gamma_grid = griddata((A_grid.flatten(), t_grid.flatten()), Gamma_vals.flatten(), (A_grid, t_grid), method='cubic')
Theta_grid = griddata((A_grid.flatten(), t_grid.flatten()), Theta_vals.flatten(), (A_grid, t_grid), method='cubic')
Omega_grid = griddata((A_grid.flatten(), t_grid.flatten()), Omega_vals.flatten(), (A_grid, t_grid), method='cubic')

# 绘制二维势能面图
def plot_greek_contour(A_grid, greek_grid, greek_name):
    plt.figure(figsize=(10, 6))
    contour = plt.contourf(A_grid, greek_grid, t_grid, levels=50, cmap="viridis")
    plt.colorbar(contour)
    plt.xlabel("Asset Price (A)")
    plt.ylabel("Time (t)")
    plt.title(f"{greek_name} vs Asset Price (A) and Time (t)")
    plt.show()

# 绘制 Delta 的二维势能面图
plot_greek_contour(A_grid, Delta_grid, 'Delta')

# 绘制 Gamma 的二维势能面图
plot_greek_contour(A_grid, Gamma_grid, 'Gamma')

# 绘制 Theta 的二维势能面图
plot_greek_contour(A_grid, Theta_grid, 'Theta')

# 绘制 Omega 的二维势能面图
plot_greek_contour(A_grid, Omega_grid, 'Omega')
